# Part 4 – LLM Powered Feature

## Track Chosen

**Track C – Model Prediction Explanation Pipeline**

In this part, the best machine learning model from Part 3 is loaded using Joblib. The model predicts the class and probability for sample inputs. These predictions are then passed to a Large Language Model (LLM), which generates structured JSON explanations. The responses are validated using a JSON schema before being accepted.

In [76]:
import os
import re
import json
import joblib
import requests

from jsonschema import validate
from jsonschema.exceptions import ValidationError

In [77]:
import os

api_key = os.environ.get("LLM_API_KEY")

if api_key is None:
    raise ValueError("LLM_API_KEY environment variable not found.")

print("API Key Loaded Successfully")

API Key Loaded Successfully


## LLM API Connection

The API key is stored in an environment variable instead of being hardcoded throughout the notebook. This improves security and follows good development practices.

In [78]:
url = "https://openrouter.ai/api/v1/chat/completions"

model_name = "openai/gpt-oss-20b:free"

def call_llm(system_prompt,
             user_prompt,
             temperature=0.0,
             max_tokens=512):

    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload
    )

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    return response.json()["choices"][0]["message"]["content"]

In [79]:
system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word: hello"

response = call_llm(
    system_prompt,
    user_prompt
)

print(response)

hello


# Load Trained Machine Learning Model

The best machine learning pipeline created in Part 3 is loaded using Joblib.

This model will be used to generate predictions and prediction probabilities for sample inputs before sending the results to the LLM for explanation.

In [80]:
loaded_model = joblib.load("best_model.pkl")

print("Best Model Loaded Successfully")

Best Model Loaded Successfully


# JSON Output Schema

The LLM is instructed to generate a structured JSON explanation.

The response will be validated using JSON Schema before it is accepted.

In [81]:
output_schema = {
    "type": "object",
    "properties": {

        "prediction_label": {
            "type": "string"
        },

        "confidence_level": {
            "type": "string"
        },

        "top_reason": {
            "type": "string"
        },

        "second_reason": {
            "type": "string"
        },

        "next_step": {
            "type": "string"
        }

    },

    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

print("Schema Created Successfully")

Schema Created Successfully


# System Prompt

The LLM is instructed to return only valid JSON without any additional explanation.

In [82]:
system_prompt = """
You are an AI assistant that explains machine learning model predictions.

Your task is to explain the prediction in a simple and structured way.

IMPORTANT RULES:

1. Return ONLY valid JSON.
2. Do NOT write any explanation outside the JSON.
3. Do NOT use markdown.
4. Do NOT use code blocks.
5. Do NOT return numeric confidence values.
6. confidence_level must be ONLY one of these:
   - low
   - medium
   - high
7. If predicted class is 1, prediction_label must be "High Price".
8. If predicted class is 0, prediction_label must be "Low Price".
9. Keep the reasons short and easy to understand.
10. next_step should contain one practical recommendation.

Return exactly this JSON format:

{
  "prediction_label": "High Price or Low Price",
  "confidence_level": "low | medium | high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}
"""

# Load Dataset and Best Model

Now I am loading the cleaned dataset and the trained model saved in Part 3. This model will be used to generate predictions before sending the information to the LLM.

In [83]:
import pandas as pd
import joblib

df = pd.read_csv("../data/cleaned_data.csv")

print("Dataset Loaded Successfully")

loaded_model = joblib.load("best_model.pkl")

print("Model Loaded Successfully")

Dataset Loaded Successfully
Model Loaded Successfully


# Prepare Feature Matrix

The target column is removed and categorical variables are converted into dummy variables so that the input format matches the trained model.

In [84]:
quality_mapping = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

X = df.drop("SalePrice", axis=1)

X["ExterQual"] = X["ExterQual"].map(quality_mapping)
X["KitchenQual"] = X["KitchenQual"].map(quality_mapping)
X["HeatingQC"] = X["HeatingQC"].map(quality_mapping)

categorical_cols = X.select_dtypes(include="object").columns
X[categorical_cols] = X[categorical_cols].fillna("None")

X = pd.get_dummies(X, drop_first=True)

print("Feature Matrix Shape :", X.shape)

Feature Matrix Shape : (1460, 254)


# Create Three Sample Inputs

Three sample records are selected from the dataset. These records will be used to generate predictions and explain the model's decision using the LLM.

In [85]:
sample_records = []

for i in range(3):
    sample_records.append(X.iloc[i].to_dict())

print("Three sample records created successfully.")

sample_records[0]

Three sample records created successfully.


{'Id': 1,
 'MSSubClass': 60,
 'LotFrontage': 65.0,
 'LotArea': 8450,
 'OverallQual': 7,
 'OverallCond': 5,
 'YearBuilt': 2003,
 'YearRemodAdd': 2003,
 'MasVnrArea': 196.0,
 'ExterQual': 4,
 'BsmtFinSF1': 706,
 'BsmtFinSF2': 0,
 'BsmtUnfSF': 150,
 'TotalBsmtSF': 856,
 'HeatingQC': 5,
 '1stFlrSF': 856,
 '2ndFlrSF': 854,
 'LowQualFinSF': 0,
 'GrLivArea': 1710,
 'BsmtFullBath': 1,
 'BsmtHalfBath': 0,
 'FullBath': 2,
 'HalfBath': 1,
 'BedroomAbvGr': 3,
 'KitchenAbvGr': 1,
 'KitchenQual': 4,
 'TotRmsAbvGrd': 8,
 'Fireplaces': 0,
 'GarageYrBlt': 2003.0,
 'GarageCars': 2,
 'GarageArea': 548,
 'WoodDeckSF': 0,
 'OpenPorchSF': 61,
 'EnclosedPorch': 0,
 '3SsnPorch': 0,
 'ScreenPorch': 0,
 'PoolArea': 0,
 'MiscVal': 0,
 'MoSold': 2,
 'YrSold': 2008,
 'MSZoning_FV': False,
 'MSZoning_RH': False,
 'MSZoning_RL': True,
 'MSZoning_RM': False,
 'Street_Pave': True,
 'Alley_None': True,
 'Alley_Pave': False,
 'LotShape_IR2': False,
 'LotShape_IR3': False,
 'LotShape_Reg': True,
 'LandContour_HLS': False

# Define JSON Output Schema

The LLM is expected to return only valid JSON. A JSON schema is defined so that every response can be validated before it is accepted.

In [86]:
import json
from jsonschema import validate, ValidationError

output_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

print("JSON Schema Created Successfully")

JSON Schema Created Successfully


# Generate Prediction for Sample Records

The trained model predicts the class label and probability for each sample record. These results will be passed to the LLM for explanation.

In [87]:
predictions = loaded_model.predict(X.iloc[:3])

probabilities = loaded_model.predict_proba(X.iloc[:3])

for i in range(3):

    print(f"\nRecord {i+1}")

    print("Predicted Class :", predictions[i])

    print("Probability :", round(probabilities[i][1], 4))


Record 1
Predicted Class : 1
Probability : 0.995

Record 2
Predicted Class : 1
Probability : 0.84

Record 3
Predicted Class : 1
Probability : 0.995


# Create User Prompt

The prediction result and probability are combined with the feature values to create the prompt that will be sent to the LLM.

In [88]:
user_prompt = f"""
House Features:
{sample_records[0]}

Predicted Class:
{predictions[0]}

Prediction Probability:
{round(probabilities[0][1],4)}

Explain this machine learning prediction.

Return ONLY valid JSON.

Do not return any extra text.

Remember:
- prediction_label should be "High Price" or "Low Price"
- confidence_level should be only "low", "medium", or "high"
"""

# Call LLM and Validate JSON Response

The response returned by the LLM is parsed as JSON and validated using the predefined schema. If the response is not valid JSON or does not match the schema, a fallback response is returned.

In [89]:
raw_response = call_llm(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    temperature=0.0,
    max_tokens=512
)

print("Raw LLM Response:\n")
print(raw_response)

Raw LLM Response:

{
  "prediction_label": "High Price",
  "confidence_level": "high",
  "top_reason": "Large living area and good overall quality boost value",
  "second_reason": "Modern features like central air and finished basement add appeal",
  "next_step": "Consider listing at a premium price and highlighting the key selling points in marketing materials"
}


In [90]:
try:
    parsed_response = json.loads(raw_response.strip())

    validate(instance=parsed_response, schema=output_schema)

    print("Validation Status : PASS")

    print(parsed_response)

except json.JSONDecodeError as e:

    print("JSON Decode Error :", e)

    parsed_response = {
        "prediction_label": None,
        "confidence_level": None,
        "top_reason": None,
        "second_reason": None,
        "next_step": None
    }

except ValidationError as e:

    print("Schema Validation Error :", e.message)

    parsed_response = {
        "prediction_label": None,
        "confidence_level": None,
        "top_reason": None,
        "second_reason": None,
        "next_step": None
    }

print("\nFinal Output")

print(parsed_response)

Validation Status : PASS
{'prediction_label': 'High Price', 'confidence_level': 'high', 'top_reason': 'Large living area and good overall quality boost value', 'second_reason': 'Modern features like central air and finished basement add appeal', 'next_step': 'Consider listing at a premium price and highlighting the key selling points in marketing materials'}

Final Output
{'prediction_label': 'High Price', 'confidence_level': 'high', 'top_reason': 'Large living area and good overall quality boost value', 'second_reason': 'Modern features like central air and finished basement add appeal', 'next_step': 'Consider listing at a premium price and highlighting the key selling points in marketing materials'}


# PII Guardrail

Before sending any input to the LLM, a regular expression is used to detect Personally Identifiable Information (PII), such as email addresses or phone numbers. If PII is detected, the request is blocked.

In [91]:
import re

def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'

    return bool(
        re.search(email_pattern, text)
        or re.search(phone_pattern, text)
    )


def safe_call_llm(system_prompt, user_prompt, temperature=0.0):

    if has_pii(user_prompt):

        print("Input blocked: PII detected.")

        return None

    return call_llm(
        system_prompt,
        user_prompt,
        temperature=temperature
    )

In [92]:
print("Test 1")

safe_call_llm(
    system_prompt,
    "My email is student@gmail.com"
)

print("\n-----------------------\n")

print("Test 2")

response = safe_call_llm(
    system_prompt,
    "This house has 3 bedrooms."
)

print(response)

Test 1
Input blocked: PII detected.

-----------------------

Test 2
{
  "prediction_label": "Low Price",
  "confidence_level": "medium",
  "top_reason": "Number of bedrooms is below the average for high-priced houses.",
  "second_reason": "Typical 3-bedroom houses are priced lower in the market.",
  "next_step": "Consider adding a fourth bedroom or upgrading amenities to increase value."
}


# Temperature Comparison

The same input is tested with temperature 0.0 and 0.7 to observe the difference in response consistency.

In [93]:
temp_comparison = []

for i in range(3):
    prompt_i = f"""
House Features:
{sample_records[i]}

Predicted Class:
{predictions[i]}

Prediction Probability:
{round(probabilities[i][1], 4)}

Return ONLY valid JSON.
"""
    out_0 = safe_call_llm(system_prompt, prompt_i, temperature=0.0)
    out_07 = safe_call_llm(system_prompt, prompt_i, temperature=0.7)

    print(f"Record {i+1}")
    print("Temperature = 0\n", out_0)
    print("\nTemperature = 0.7\n", out_07)
    print("="*50)

    temp_comparison.append({
        "Record": i + 1,
        "Temp=0": out_0,
        "Temp=0.7": out_07
    })

pd.DataFrame(temp_comparison)

Record 1
Temperature = 0
 {
  "prediction_label": "High Price",
  "confidence_level": "high",
  "top_reason": "Large living area and good overall quality boost value",
  "second_reason": "Well-maintained condition and modern features add appeal",
  "next_step": "Consider a professional appraisal to confirm market value before listing"
}

Temperature = 0.7
 {
  "prediction_label": "High Price",
  "confidence_level": "high",
  "top_reason": "Large living area of 1710 sq ft",
  "second_reason": "High overall quality rating of 7",
  "next_step": "List the property at a premium price and highlight its high quality features"
}
Record 2
Temperature = 0
 {
  "prediction_label": "High Price",
  "confidence_level": "high",
  "top_reason": "Large living area and good overall condition boost value",
  "second_reason": "Solid foundation and well-maintained features add appeal",
  "next_step": "Prepare a premium listing with professional photos to attract buyers"
}

Temperature = 0.7
 {
  "predictio

,Record,Temp=0,Temp=0.7
0,1,"{\n ""prediction_label"": ""High Price"",\n ""con...","{\n ""prediction_label"": ""High Price"",\n ""con..."
1,2,"{\n ""prediction_label"": ""High Price"",\n ""con...","{\n ""prediction_label"": ""High Price"",\n ""con..."
2,3,"{\n ""prediction_label"": ""High Price"",\n ""con...","{\n ""prediction_label"": ""High Price"",\n ""con..."


# Complete Prediction Explanation Pipeline

The complete pipeline is executed for three sample records. Each record is passed through the trained model, explained by the LLM, parsed into JSON, validated against the schema, and the validation status is reported.

In [94]:
results = []

for i in range(3):

    feature_values = sample_records[i]

    prediction = predictions[i]

    probability = round(probabilities[i][1], 4)

    prompt = f"""
House Features:
{feature_values}

Predicted Class:
{prediction}

Prediction Probability:
{probability}

Return ONLY valid JSON.
"""

    raw = safe_call_llm(system_prompt, prompt, temperature=0.0)

    if raw is None:
        results.append({
            "Record": i + 1,
            "Prediction": "Blocked",
            "Probability": "-",
            "Validation": "Blocked (PII)"
        })
        continue

    try:
        result = json.loads(raw.strip())
        validate(instance=result, schema=output_schema)

        results.append({
            "Record": i + 1,
            "Prediction": result["prediction_label"],
            "Probability": probability,
            "Validation": "PASS"
        })

    except Exception:
        results.append({
            "Record": i + 1,
            "Prediction": "Invalid",
            "Probability": probability,
            "Validation": "FAIL"
        })

results_df = pd.DataFrame(results)

results_df

,Record,Prediction,Probability,Validation
0,1,High Price,0.995,PASS
1,2,High Price,0.840,PASS
2,3,High Price,0.995,PASS
